# Function 6: Cake and Stuff
Time to get cooking! You are optimising a cake recipe. There are five ingredients. The outputs correspond to the sum of different objectives: flavor, consistency, calories, waste and cost. Each objective receives negative points by our expert taster. You want this sum to be as close to zero as possible!

In [4]:
import numpy as np
from scipy.optimize import minimize
from scipy.stats import norm
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel, ConstantKernel as C
import matplotlib.pyplot as plt


In [5]:
def load_inputs(file_path):
    with open(file_path, "r") as f:
        content = f.read()

    # Make the file a proper list of lists
    content = "[" + content.replace("]\n[", "],[") + "]"

    # Safe eval with restricted globals
    return eval(content, {"array": np.array})


def load_outputs(file_path):
    with open(file_path, "r") as f:
        content = f.read()

    content = "[" + content.replace("]\n[", "],[") + "]"

    return eval(content, {"np": np})


def get_input_points(function_number, file_path="../inputs.txt"):
    if not 1 <= function_number <= 8:
        raise ValueError("Function number must be between 1 and 8")

    data = load_inputs(file_path)
    index = function_number - 1

    inputs = [dataset[index] for dataset in data]
    return np.array(inputs)


def get_output_points(function_number, file_path="../outputs.txt"):
    if not 1 <= function_number <= 8:
        raise ValueError("Function number must be between 1 and 8")

    data = load_outputs(file_path)
    index = function_number - 1

    outputs = [row[index] for row in data]
    return np.array(outputs)

# Load inputs
X = np.load(r'initial_inputs.npy')
y = np.load(r'initial_outputs.npy')


# Get input and outputs from submissions
inputs_array = get_input_points(6)
outputs_array = get_output_points(6)


# Append inputs_f1_array to X
X = np.vstack((X, inputs_array))

# Append outputs_f1_array to Y
y = np.hstack((y, outputs_array))
y = y.ravel()

print("New shape of X:", X.shape)
print("New shape of Y:", y.shape)

New shape of X: (29, 5)
New shape of Y: (29,)


In [6]:
# ----- Flatten outputs -----
y = y.ravel()

# ----- Fit GP -----
kernel = C(1.0, (1e-3, 1e5)) * Matern(length_scale=np.ones(X.shape[1]), nu=2.5) + WhiteKernel(noise_level=1e-8)
gp = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=20, normalize_y=True)
gp.fit(X, y)

# ----- Expected Improvement -----
def expected_improvement(x, gp, y_best, xi=0.01):
    x = np.atleast_2d(x)
    mean, std = gp.predict(x, return_std=True)
    mean = mean.ravel()
    std = std.ravel() + 1e-9
    improvement = y_best - mean - xi
    Z = improvement / std
    EI = improvement * norm.cdf(Z) + std * norm.pdf(Z)
    return -EI  # minimize negative EI

# ----- Current best recipe -----
X_best = X[np.argmin(y)]
y_best = y.min()

# ----- Bounds with slight interior bias -----
bounds = [(0.01, 0.99)]*X.shape[1]  # avoids boundary hits

# ----- Local refinement around best recipe -----
perturbations = 0.05 * np.random.rand(20, X.shape[1])
initial_points = np.clip(X_best + perturbations, 0.01, 0.99)

# Add some global exploration points
global_points = np.random.uniform(0.01, 0.99, size=(10, X.shape[1]))
initial_points = np.vstack([initial_points, global_points])

best_x = None
best_ei = float("inf")
for x0 in initial_points:
    res = minimize(expected_improvement, x0=x0, bounds=bounds, args=(gp, y_best), method="L-BFGS-B")
    if res.fun < best_ei:
        best_ei = res.fun
        best_x = res.x

# ----- Round and format next candidate -----
next_query = np.round(best_x, 6)
formatted_next_query = f"{next_query[0]:.6f}-{next_query[1]:.6f}-{next_query[2]:.6f}-{next_query[3]:.6f}-{next_query[4]:.6f}"
print("Next Query Point:", formatted_next_query)

/opt/anaconda3/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


Next Query Point: 0.175081-0.990000-0.052187-0.230337-0.808562
